In [46]:
import os
from langchain_cerebras import ChatCerebras
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from pydantic import BaseModel
from pymongo import MongoClient
import sib_api_v3_sdk
from sib_api_v3_sdk.rest import ApiException
from langchain.agents import create_agent
from langchain_groq import ChatGroq

load_dotenv()

CEREBRAS_API_KEY=os.getenv("CEREBRAS_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
BREVO_API_KEY=os.getenv("BREVO_API_KEY")
MONGODB_URI=os.getenv("MONGODB_URI")


In [47]:
DB_NAME = "soulsync"
COLLECTION_NAME="users"
SENDER_NAME="SoulSync"
SENDER_MAIL="lokeshvijayraina@gmail.com"


In [48]:
llm_gpt=ChatGroq(model="openai/gpt-oss-120b",api_key=GROQ_API_KEY)
client= MongoClient(MONGODB_URI)
users_collection=client[DB_NAME][COLLECTION_NAME]

In [49]:
class EmailTemplate(BaseModel):
    subject_template: str
    body_template: str
    reason: str

class EmailDraft(BaseModel):
    recipient: str
    subject: str
    body: str

class DispatchedResult(BaseModel):
    totalUsers: int
    sent: int
    failed: int


In [50]:
@tool
def find_users(
    filters: dict = None,
    sort_by: str = None,
    ascending: bool = False,
    limit: int = 5,
):
    """
    Query SoulSync users with optional filters, sorting, and limit.

    Available fields:
    - name
    - email
    - totalListeningTime
    - updatedAt (last active)
    - createdAt (joined date)
    """

    query = filters or {}

    targeted_users = users_collection.find(
        query,
        {
            "_id": 0,
            "name": 1,
            "email": 1,
            "totalListeningTime": 1,
            "updatedAt": 1,
            "createdAt": 1,
        },
    )

    if sort_by:
        targeted_users = targeted_users.sort(
            sort_by,
            -1 if not ascending else 1,
        )

    targeted_users = targeted_users.limit(limit)

    return list(targeted_users)

In [51]:
finder_agent=create_agent(
    model=llm_gpt,
    tools=[find_users],
    system_prompt=
    """
    You are LookOut's user discovery agent.

Your job is to identify the correct SoulSync users by calling the `find_users` tool.

Available fields:

* `name`
* `email`
* `country`
* `totalListeningTime` (seconds)
* `createdAt` (join date)
* `updatedAt` (last active where the time is in the UTC soo add 5:30 and show in the indian railway time)

Infer these arguments from the user's request:

* `filters`
* `sort_by`
* `ascending`
* `limit`

Intent examples:

* Top / most active → `sort_by="totalListeningTime"`, `ascending=False`
* Least active → `sort_by="totalListeningTime"`, `ascending=True`
* Newest → `sort_by="createdAt"`, `ascending=False`
* Oldest → `sort_by="createdAt"`, `ascending=True`
* Recently active → `sort_by="updatedAt"`, `ascending=False`
* Inactive → `sort_by="updatedAt"`, `ascending=True`
* Country-specific requests → use `filters`

Rules:

* Always call `find_users`.
* Never invent users or fields.
* Combine filters and sorting when needed.
* If the request cannot be answered using the available fields, explain why.

    """)

In [52]:
template_llm=llm_gpt.with_structured_output(EmailTemplate)

def generate_template(sample_user: dict, intent: str) -> EmailTemplate:
    prompt = f"""
Product: SoulSync
User intent: {intent}
Sample user fields available: {list(sample_user.keys())}
Write ONE email template with placeholders like {{name}}, {{minutesListened}}, {{rank}}.
Only reference fields given. Sign off as "The SoulSync Team". Keep body under 100 words.
"""
    return template_llm.invoke(prompt)


In [53]:
def renderTemplate(template: EmailTemplate, user: dict) -> EmailDraft:
    subject = template.subject_template.format(**user)
    body = template.body_template.format(**user)
    return EmailDraft(recipient=user["email"], subject=subject, body=body)


In [54]:
@tool
def sendMail(receiver: str, subject: str, body: str) -> str:
    """Send an approved email using Brevo."""
    configuration = sib_api_v3_sdk.Configuration()
    configuration.api_key["api-key"] = BREVO_API_KEY
    api_instance = sib_api_v3_sdk.TransactionalEmailsApi(sib_api_v3_sdk.ApiClient(configuration))
    send_smtp_email = sib_api_v3_sdk.SendSmtpEmail(
        sender={"name": SENDER_NAME, "email": SENDER_MAIL},
        to=[{"email": receiver}],
        subject=subject,
        html_content=body,
    )
    try:
        api_response = api_instance.send_transac_email(send_smtp_email)
        return f"Sent to {receiver} (id: {api_response.message_id})"
    except ApiException as e:
        return f"Failed: {e}"

In [55]:
import ast

def get_tool_result(agent_response):
    for msg in agent_response["messages"]:
        if hasattr(msg, "name") and msg.name == "find_users":
            return ast.literal_eval(msg.content)
    return None


In [ ]:
def run_dispatch(user_prompt: str):
    response = finder_agent.invoke({"messages": [HumanMessage(content=user_prompt)]})
    matched_users = get_tool_result(response)

    for user in matched_users:
        user["minutesListened"] = round(user.pop("totalListeningTime", 0) / 60)

    template = generate_template(matched_users[0], user_prompt)
    preview = renderTemplate(template, matched_users[0])

    print(f"Matched {len(matched_users)} users.")
    print(f"Subject: {preview.subject}\n\n{preview.body}\n")
    choice = input("Approve this template for all matched users? (y/n): ").strip().lower()

    if choice != "y":
        print("Aborted.")
        return DispatchedResult(totalUsers=len(matched_users), sent=0, failed=0)

    sent, failed = 0, 0
    for user in matched_users:
        draft = renderTemplate(template, user)
        result = sendMail.invoke({"receiver": draft.recipient, "subject": draft.subject, "body": draft.body})
        if "Failed" in result:
            failed += 1
        else:
            sent += 1

    return DispatchedResult(totalUsers=len(matched_users), sent=sent, failed=failed)


In [ ]:
response = finder_agent.invoke({"messages": [HumanMessage(content="mail our the top 1 listenedtime user")]})
for msg in response["messages"]:
    print(type(msg), getattr(msg, "name", None))
    print(repr(msg.content)[:500])
    print("---")


<class 'langchain_core.messages.human.HumanMessage'> None
'mail our the top 1 listenedtime user'
---
<class 'langchain_core.messages.ai.AIMessage'> None
''
---
<class 'langchain_core.messages.tool.ToolMessage'> find_users
"[{'createdAt': datetime.datetime(2026, 3, 4, 8, 44, 49, 307000), 'email': 'lokeshvijayraina@gmail.com', 'name': 'Loki', 'totalListeningTime': 472656, 'updatedAt': datetime.datetime(2026, 7, 6, 9, 25, 33, 539000)}]"
---
<class 'langchain_core.messages.ai.AIMessage'> None
'**Top listener (by total listening time)**  \n\n| Name | Email | Total Listening Time (seconds) | Joined (createdAt) | Last Active (updatedAt – IST) |\n|------|-------|--------------------------------|--------------------|------------------------------|\n| Loki | lokeshvijayraina@gmail.com | 472\u202f656 | 2026‑03‑04\u202f08:44:49 (UTC) | 2026‑07‑06\u202f14:55:33 (IST, UTC+5:30) |\n\nThis is the user with the highest `totalListeningTime` in the system.'
---


In [ ]:
result = run_dispatch("mail our the top 1 listenedtime user")
print(result)

ValueError: malformed node or string on line 1: <ast.Call object at 0x7f634789ba50>